In [1]:
%run common_imports.py

%matplotlib qt
%config InlineBackend.figure_format = 'retina'
sns.set_context("talk")

%reload_ext autoreload
%autoreload 2
pd.options.display.max_rows = 600
pd.set_option('display.float_format', lambda x: '%.9f' % x)

dj.config['display.limit'] = 10**3  

os.environ["SPYGLASS_USE_TRANSACTIONS"] = "1"  
os.environ['KACHERY_API_KEY'] = "RhysjLwgmBAt2ObCyXXaDnqAv2kTdYRa"

[2026-03-12 11:51:19,772][INFO]: DataJoint is configured from /media/labuser/NA_1_2025/spyglass/wilbur/dj_local_conf.json
[2026-03-12 11:51:20,188][INFO]: DataJoint 0.14.9 connected to anirudh@172.16.102.154:3306


In [43]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats         
from scipy.stats import chi2 
import spyglass.linearization.v1 as sgpl                                                                       

### Load data

In [4]:
#extract position data
#read from csv
trialized_position = pd.read_csv("/media/labuser/NA_1_2025/spyglass/wilbur/analysis/position/trialized_position.csv", index_col = "time")

In [5]:
#extract spikes
#read from npz
data = np.load("/media/labuser/NA_1_2025/spyglass/wilbur/analysis/final_spikes/mfpc_spikes.npz", allow_pickle=True)
mpfc_spikes = [data[f"arr_{i}"] for i in range(len(data.files))]


### Prepare dataframe for regression

In [6]:
def fit_glm_all_units(formula: str,
                      cov_df: pd.DataFrame,
                      spike_counts_masked: np.array,
                      unit_ids: np.array,
                      bin_size = 0.002):
    
    rows = []
    for i, uid in enumerate(unit_ids):
        df = cov_df.copy()
        df["spike_count"] = spike_counts_masked[i]  # pre-masked counts
        try:
            res = smf.glm(formula, data=df, family=sm.families.Poisson()).fit(disp=False)
            rows.append(dict(                                                                                       
                unit=uid,   
                aic=res.aic,
                llf=res.llf,
                deviance=res.deviance,
                n_params=len(res.params),
                n_obs=int(res.nobs),
                converged=res.converged,
                coef=res.params.to_dict(),
                bse=res.bse.to_dict(),
                deviance_null = res.null_deviance,
                df_model = res.df_model
            ))

        except Exception as e:
            rows.append(dict(
                unit=uid, aic=np.nan, llf=np.nan, deviance=np.nan,
                n_params=np.nan, n_obs=np.nan, converged=False,
                coef=None, bse=None, deviance_null=np.nan,
                df_model=np.nan, error=str(e)          
            ))


    return pd.DataFrame(rows)

In [7]:
BIN_SIZE = 0.002  

bin_edges = np.arange(
    trialized_position.index.min(), trialized_position.index.max() + BIN_SIZE, BIN_SIZE
)
bin_centers = bin_edges[:-1] + BIN_SIZE / 2

spike_counts = np.array([np.histogram(spikes, bins=bin_edges)[0] for spikes in mpfc_spikes]) 

In [8]:
def interp_col(col_values, times, bin_centers):
    if pd.api.types.is_numeric_dtype(col_values):
        return np.interp(bin_centers, times, col_values.astype(float))
    else:
        idx = np.searchsorted(times, bin_centers).clip(0, len(times) - 1)
        return col_values.iloc[idx].values

cols_to_interp = [c for c in trialized_position.columns if c != "video_frame_ind"]
times = trialized_position.index.astype(float).values

interpolated = {col: interp_col(trialized_position[col], times, bin_centers) for col in cols_to_interp}

interp_trialised_position = pd.DataFrame(interpolated, columns=cols_to_interp)
interp_trialised_position.insert(0, "time_bin_center", bin_centers)

mask = (interp_trialised_position["zone"]=="run") &\
    (interp_trialised_position["trial_type"].isin(["outbound", "inbound"]))


cov_df = interp_trialised_position[mask]
spike_counts_masked = spike_counts[:, mask]
unit_ids = np.arange(0, len(spike_counts_masked))

# print(cov_df.head(1))
# print(spike_counts_masked[0].shape)
#print(unit_ids)

In [9]:
cov_df = cov_df.rename(columns={"left/right": "choice"})

In [10]:
# Intersection of all predictor filters — all models fit on this for valid AIC comparison
common_mask = (
    cov_df["speed"].notna() & (cov_df["speed"] > 5) & (cov_df["speed"] < 120)
    & cov_df["linear_position"].notna()
)
cov_df_common = cov_df[common_mask].copy()
spike_counts_common = spike_counts_masked[:, common_mask]

# Scaling params defined once — used by all single-unit and all-units cells
speed_min_val = cov_df_common["speed"].min()
speed_max_val = cov_df_common["speed"].max()
pos_min_val   = cov_df_common["linear_position"].min()
pos_max_val   = cov_df_common["linear_position"].max()

cov_df_common["speed_scaled"] = (cov_df_common["speed"] - speed_min_val) / (speed_max_val - speed_min_val)
cov_df_common["pos_scaled"]   = (cov_df_common["linear_position"] - pos_min_val) / (pos_max_val - pos_min_val)

# Outbound-only common subset — baseline and choice model
outbound_common_mask = common_mask & (cov_df["trial_type"] == "outbound")
cov_df_out_common = cov_df[outbound_common_mask].copy()
spike_counts_out_common = spike_counts_masked[:, outbound_common_mask]
cov_df_out_common["speed_scaled"] = (cov_df_out_common["speed"] - speed_min_val) / (speed_max_val - speed_min_val)
cov_df_out_common["pos_scaled"]   = (cov_df_out_common["linear_position"] - pos_min_val) / (pos_max_val - pos_min_val)

print(f"cov_df_common: {len(cov_df_common):,} bins")
print(f"cov_df_out_common: {len(cov_df_out_common):,} bins (outbound only)")

cov_df_common: 1,132,036 bins
cov_df_out_common: 194,672 bins (outbound only)


### Models:

#### Single variable models:
1. Null model (constant rate)
2. Null model (outbound only, for comparison with other outbound-only models)
2. spike_count ~ trial_type (categorical)
3. spike_count ~ left/right choice (categorical)
4. spike_count ~ speed (linear)
5. spike_count ~ bs(speed, df = 4) (spline)
6. spike_count ~ bs(linear_position, df = 8) (spline)

#### Mutli-variable models:
1. 

### Null model

#### Fit on one unit 

In [13]:
unit_idx = 9
spk_cov_df = cov_df.copy()
spk_cov_df["spike_count"] = spike_counts_masked[unit_idx]

spk_cov_df_common = cov_df_common.copy()
spk_cov_df_common["spike_count"] = spike_counts_common[unit_idx]

In [14]:
model_constant = smf.glm("spike_count ~ 1", data=spk_cov_df, family=sm.families.Poisson())
results_constant = model_constant.fit()

print(results_constant.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1835243
Model:                            GLM   Df Residuals:                  1835242
Model Family:                 Poisson   Df Model:                            0
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -15493.
Date:                Thu, 12 Mar 2026   Deviance:                       27031.
Time:                        11:54:45   Pearson chi2:                 1.83e+06
No. Iterations:                     8   Pseudo R-squ. (CS):              0.000
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -6.8328      0.022   -303.889      0.0

In [15]:
# Interpret the coefficient
mean_count_per_bin = np.exp(results_constant.params["Intercept"])
mean_rate_hz = mean_count_per_bin / BIN_SIZE

print(f"β₀ = {results_constant.params['Intercept']:.4f}")
print(f"exp(β₀) = {mean_count_per_bin:.4f} spikes/bin")
print(f"Firing rate = {mean_rate_hz:.2f} Hz")
print(f"Observed mean = {spk_cov_df['spike_count'].mean():.4f} spikes/bin")

β₀ = -6.8328
exp(β₀) = 0.0011 spikes/bin
Firing rate = 0.54 Hz
Observed mean = 0.0011 spikes/bin


#### Fit on all units

In [ ]:
# null_model_all = fit_glm_all_units("spike_count ~ 1", cov_df_common, spike_counts_common, unit_ids)

In [16]:
# null_model_all["model"] = "null"
# null_model_all.to_csv(f"{base_dir}/analysis/null_model_all.csv")
# keep_default_na=False prevents pandas reading "null" string as NaN
null_model_all = pd.read_csv(f"{base_dir}/analysis/null_model_all.csv", index_col=0,
                              keep_default_na=False, na_values=[''])
null_model_all["model"] = "null"  # re-assign after load in case CSV was saved without it

In [17]:
null_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,121428.789710014,-60713.394855007,100002.515597236,1.000000000,1132036.000000000,True,{'Intercept': -4.659102359814778},{'Intercept': 0.00965563990073221},100002.515597235,0.000000000,NaN,null
1,1,50660.859049688,-25329.429524844,43103.631638410,1.000000000,1132036.000000000,True,{'Intercept': -5.702313636012289},{'Intercept': 0.016267153093296714},43103.631638410,0.000000000,NaN,null
2,2,25010.949533751,-12504.474766875,21684.949533751,1.000000000,1132036.000000000,True,{'Intercept': -6.523751363946384},{'Intercept': 0.02452926043834059},21684.949533751,0.000000000,NaN,null
3,3,73355.428758445,-36676.714379222,61652.360230250,1.000000000,1132036.000000000,True,{'Intercept': -5.264647872107706},{'Intercept': 0.013069941416749423},61652.360230251,0.000000000,NaN,null
4,4,101430.882864172,-50714.441432086,84168.882864171,1.000000000,1132036.000000000,True,{'Intercept': -4.876528555283104},{'Intercept': 0.01076451464846956},84168.882864172,0.000000000,NaN,null


### Null (outbound only)

#### Fit on one unit

In [18]:
model_constant_out = smf.glm("spike_count ~ 1", 
                             data=spk_cov_df[spk_cov_df["trial_type"]=="outbound"], 
                             family=sm.families.Poisson())
results_constant_out = model_constant_out.fit()

print(results_constant_out.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               867583
Model:                            GLM   Df Residuals:                   867582
Model Family:                 Poisson   Df Model:                            0
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -4094.8
Date:                Thu, 12 Mar 2026   Deviance:                       7225.7
Time:                        11:55:16   Pearson chi2:                 8.67e+05
No. Iterations:                     9   Pseudo R-squ. (CS):              0.000
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -7.4955      0.046   -164.560      0.0

In [19]:
# Interpret the coefficient
mean_count_per_bin = np.exp(results_constant_out.params["Intercept"])
mean_rate_hz = mean_count_per_bin / BIN_SIZE

print(f"β₀ = {results_constant_out.params['Intercept']:.4f}")
print(f"exp(β₀) = {mean_count_per_bin:.4f} spikes/bin")
print(f"Firing rate = {mean_rate_hz:.2f} Hz")
print(f'Observed mean = {spk_cov_df[spk_cov_df["trial_type"]=="outbound"]["spike_count"].mean():.4f} spikes/bin')

β₀ = -7.4955
exp(β₀) = 0.0006 spikes/bin
Firing rate = 0.28 Hz
Observed mean = 0.0006 spikes/bin


#### Fit on all units

In [ ]:
# null_model_out_all = fit_glm_all_units("spike_count ~ 1", cov_df_out_common, spike_counts_out_common, unit_ids)

In [20]:
# null_model_out_all["model"] = "null_out"
# null_model_out_all.to_csv(f"{base_dir}/analysis/null_model_out_all.csv")
# keep_default_na=False prevents pandas reading "null" string as NaN (fixed)
null_model_out_all = pd.read_csv(f"{base_dir}/analysis/null_model_out_all.csv", index_col=0,
                                  keep_default_na=False, na_values=[''])
null_model_out_all.head(2)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,43163.862357804,-21580.931178902,34104.180123971,1.000000000,194672.000000000,True,{'Intercept': -3.759932119086138},{'Intercept': 0.014852759209998779},34104.180123971,0.000000000,NaN,null_out
1,1,19967.448301989,-9982.724150994,16474.220890711,1.000000000,194672.000000000,True,{'Intercept': -4.7134160598930235},{'Intercept': 0.023925084946465218},16474.220890711,0.000000000,NaN,null_out


### Trial type

#### Fit on one unit

In [21]:
model_trial_type = smf.glm("spike_count ~ trial_type", data=spk_cov_df, family=sm.families.Poisson())
results_trial_type = model_trial_type.fit()

print(results_trial_type.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1835243
Model:                            GLM   Df Residuals:                  1835241
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -15273.
Date:                Thu, 12 Mar 2026   Deviance:                       26590.
Time:                        11:57:01   Pearson chi2:                 1.83e+06
No. Iterations:                     9   Pseudo R-squ. (CS):          0.0002400
Covariance Type:            nonrobust                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -6

In [22]:
rate_inbound  = np.exp(results_trial_type.params["Intercept"]) / BIN_SIZE        # Hz
rate_outbound = np.exp(results_trial_type.params["Intercept"] + results_trial_type.params["trial_type[T.outbound]"]) / BIN_SIZE
ratio         = np.exp(results_trial_type.params["trial_type[T.outbound]"])      # outbound/inbound rate ratio

print("inbound rate: ", rate_inbound)
print("outbound rate: ", rate_outbound)
print("outbound/inbound: ", ratio)

inbound rate:  0.7729987805818047
outbound rate:  0.2777832207690415
outbound/inbound:  0.3593579029451577


#### Fit all units 

In [ ]:
# trial_type_model_all = fit_glm_all_units("spike_count ~ trial_type", cov_df_common, spike_counts_common, unit_ids)

In [23]:
# trial_type_model_all.to_csv(f"{base_dir}/analysis/trial_type_model_all.csv")
trial_type_model_all = pd.read_csv(f"{base_dir}/analysis/trial_type_model_all.csv", index_col=0)
trial_type_model_all["model"] = "trial_type"

In [24]:
trial_type_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,117744.680877903,-58870.340438952,96316.406765126,2.000000000,1132036.000000000,True,"{'Intercept': -5.0196520586988855, 'trial_type...","{'Intercept': 0.012707188138566833, 'trial_typ...",100002.515597235,1.000000000,NaN,trial_type
1,1,48962.232179403,-24479.116089701,41403.004768125,2.000000000,1132036.000000000,True,"{'Intercept': -6.1340511509645115, 'trial_type...","{'Intercept': 0.022183912735354593, 'trial_typ...",43103.631638410,1.000000000,NaN,trial_type
2,2,23838.644589440,-11917.322294720,20510.644589440,2.000000000,1132036.000000000,True,"{'Intercept': -7.118825182270768, 'trial_type[...","{'Intercept': 0.03629770043178369, 'trial_type...",21684.949533751,1.000000000,NaN,trial_type
3,3,71310.241176111,-35653.120588056,59605.172647917,2.000000000,1132036.000000000,True,"{'Intercept': -5.62934658490596, 'trial_type[T...","{'Intercept': 0.017236256333198044, 'trial_typ...",61652.360230251,1.000000000,NaN,trial_type
4,4,98260.403902078,-49128.201951039,80996.403902078,2.000000000,1132036.000000000,True,"{'Intercept': -5.253428395568738, 'trial_type[...","{'Intercept': 0.014282799726013456, 'trial_typ...",84168.882864172,1.000000000,NaN,trial_type


### Choice

#### Fit on one unit

In [25]:
choice_mask = cov_df["trial_type"]=="outbound"
choice_spk_cov_df = spk_cov_df[choice_mask]
model_choice= smf.glm("spike_count ~ choice", data=choice_spk_cov_df, family=sm.families.Poisson())
results_choice = model_choice.fit()

print(results_choice.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               867583
Model:                            GLM   Df Residuals:                   867581
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -4007.2
Date:                Thu, 12 Mar 2026   Deviance:                       7050.5
Time:                        11:57:16   Pearson chi2:                 8.67e+05
No. Iterations:                    10   Pseudo R-squ. (CS):          0.0002019
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -6.4212      0.076    -

In [26]:
rate_left  = np.exp(results_choice.params["Intercept"]) / BIN_SIZE        # Hz
rate_right = np.exp(results_choice.params["Intercept"] + results_choice.params["choice[T.right]"]) / BIN_SIZE
ratio         = np.exp(results_choice.params["choice[T.right]"])      # outbound/inbound rate ratio

print("left rate: ", rate_left)
print("right rate: ", rate_right)
print("right/left: ", ratio)

left rate:  0.8133174791934886
right rate:  0.203945659961309
right/left:  0.2507577485775272


#### Fit for all units

In [ ]:
# choice_model_all = fit_glm_all_units("spike_count ~ choice", cov_df_out_common, spike_counts_out_common, unit_ids)

In [27]:
# choice_model_all.to_csv(f"{base_dir}/analysis/choice_model_all.csv")
choice_model_all = pd.read_csv(f"{base_dir}/analysis/choice_model_all.csv", index_col=0)
choice_model_all["model"] = "choice"

In [28]:
choice_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,42787.551887263,-21391.775943632,33725.869653430,2.000000000,194672.000000000,True,"{'Intercept': -3.5172201338373696, 'choice[T.r...","{'Intercept': 0.018248296695927622, 'choice[T....",34104.180123971,1.000000000,NaN,choice
1,1,19487.988330142,-9741.994165071,15992.760918864,2.000000000,194672.000000000,True,"{'Intercept': -4.315246945217407, 'choice[T.ri...","{'Intercept': 0.027196414660878052, 'choice[T....",16474.220890711,1.000000000,NaN,choice
2,2,11208.549272800,-5602.274636400,9398.549272800,2.000000000,194672.000000000,True,"{'Intercept': -4.94256206292855, 'choice[T.rig...","{'Intercept': 0.037216146176141184, 'choice[T....",9704.267962798,1.000000000,NaN,choice
3,3,26660.525861463,-13328.262930731,21683.298450185,2.000000000,194672.000000000,True,"{'Intercept': -4.283937507563931, 'choice[T.ri...","{'Intercept': 0.0267739768910865, 'choice[T.ri...",21700.093672297,1.000000000,NaN,choice
4,4,35554.223516030,-17775.111758015,28094.223516030,2.000000000,194672.000000000,True,"{'Intercept': -3.507939324761683, 'choice[T.ri...","{'Intercept': 0.018163813014194518, 'choice[T....",29491.791912117,1.000000000,NaN,choice


### Speed

#### Fit for one unit

In [29]:
spk_cov_df_speed = spk_cov_df_common.copy()  # single-unit speed model (already speed-filtered via common_mask)

model_speed = smf.glm("spike_count ~ speed", data=spk_cov_df_common, family=sm.families.Poisson())
results_speed = model_speed.fit()

print(results_speed.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1132036
Model:                            GLM   Df Residuals:                  1132034
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -11060.
Date:                Thu, 12 Mar 2026   Deviance:                       18861.
Time:                        11:57:31   Pearson chi2:                 1.10e+06
No. Iterations:                    10   Pseudo R-squ. (CS):           0.002169
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -8.3013      0.060   -139.138      0.0

In [30]:
#interpret coefficients
beta0_speed = results_speed.params["Intercept"]
beta1_speed = results_speed.params["speed"]

print("Model interpretation:")
print(f"  β₀ = {beta0_speed:.4f} (log rate at speed=0)")
print(f"  β₁ = {beta1_speed:.5f} (change in log rate per cm/s)")
print()
print(f"At speed=0: rate = {np.exp(beta0_speed) / BIN_SIZE:.2f} Hz")
print(f"At speed=20: rate = {np.exp(beta0_speed + beta1_speed * 20) / BIN_SIZE:.2f} Hz")
print(f"Effect: {100 * (np.exp(beta1_speed) - 1):.2f}% change per 1 cm/s")

Model interpretation:
  β₀ = -8.3013 (log rate at speed=0)
  β₁ = 0.03457 (change in log rate per cm/s)

At speed=0: rate = 0.12 Hz
At speed=20: rate = 0.25 Hz
Effect: 3.52% change per 1 cm/s


#### Fit for all units

In [ ]:
# speed_model_all = fit_glm_all_units("spike_count ~ speed", cov_df_common, spike_counts_common, unit_ids)

In [31]:
# speed_model_all["model"] = "speed"
# speed_model_all.to_csv(f"{base_dir}/analysis/speed_model_all.csv")
speed_model_all = pd.read_csv(f"{base_dir}/analysis/speed_model_all.csv", index_col=0)

In [32]:
speed_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,115451.235134141,-57723.617567070,94022.961021363,2.000000000,1132036.000000000,True,"{'Intercept': -5.531505525392574, 'speed': 0.0...","{'Intercept': 0.01741446751010367, 'speed': 0....",100002.515597235,1.000000000,NaN,speed
1,1,48057.687029670,-24026.843514835,40498.459618393,2.000000000,1132036.000000000,True,"{'Intercept': -6.701143112210031, 'speed': 0.0...","{'Intercept': 0.03056836876635762, 'speed': 0....",43103.631638410,1.000000000,NaN,speed
2,2,23998.251519899,-11997.125759949,20670.251519899,2.000000000,1132036.000000000,True,"{'Intercept': -7.447585899666322, 'speed': 0.0...","{'Intercept': 0.04498287310584487, 'speed': 0....",21684.949533751,1.000000000,NaN,speed
3,3,70025.603748604,-35010.801874302,58320.535220409,2.000000000,1132036.000000000,True,"{'Intercept': -6.148447174113083, 'speed': 0.0...","{'Intercept': 0.023659348221290576, 'speed': 0...",61652.360230251,1.000000000,NaN,speed
4,4,95259.836254807,-47627.918127404,77995.836254807,2.000000000,1132036.000000000,True,"{'Intercept': -5.8997085845259045, 'speed': 0....","{'Intercept': 0.02038951377984152, 'speed': 0....",84168.882864172,1.000000000,NaN,speed


### Speed (spline)

#### FIt on one unit


In [33]:
from patsy import bs, cr

# speed_min_val, speed_max_val defined in common mask cell
bs_df = 4
model_speed_spline = smf.glm(f"spike_count ~ bs(speed_scaled, df={bs_df})",
                              data=spk_cov_df_common,
                              family=sm.families.Poisson())
results_speed_spline = model_speed_spline.fit()
print(results_speed_spline.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1132036
Model:                            GLM   Df Residuals:                  1132031
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -10940.
Date:                Thu, 12 Mar 2026   Deviance:                       18623.
Time:                        11:57:46   Pearson chi2:                 1.08e+06
No. Iterations:                    10   Pseudo R-squ. (CS):           0.002380
Covariance Type:            nonrobust                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

In [34]:
speeds_of_interest = [10, 20, 30, 40, 60]
speeds_scaled = (np.array(speeds_of_interest) - speed_min_val) / (speed_max_val - speed_min_val)
rates = results_speed_spline.predict(pd.DataFrame({"speed_scaled": speeds_scaled})) / BIN_SIZE

print("Spline model — predicted rates:")
for s, r in zip(speeds_of_interest, rates):
    print(f"  speed={s:2f} cm/s → {r:.2f} Hz")

speed_range = np.linspace(speed_min_val, speed_max_val, 500)
speed_range_scaled = (speed_range - speed_min_val) / (speed_max_val - speed_min_val)
pred_curve = results_speed_spline.predict(pd.DataFrame({"speed_scaled": speed_range_scaled})) / BIN_SIZE
peak_speed = speed_range[np.argmax(pred_curve)]
print(f"\nPeak firing at: {peak_speed:.1f} cm/s ({pred_curve.max():.2f} Hz)")
print(f"Rate ratio high/low speed: {pred_curve.max() / pred_curve.min():.2f}x")

Spline model — predicted rates:
  speed=10.000000 cm/s → 0.09 Hz
  speed=20.000000 cm/s → 0.20 Hz
  speed=30.000000 cm/s → 0.39 Hz
  speed=40.000000 cm/s → 0.63 Hz
  speed=60.000000 cm/s → 1.20 Hz

Peak firing at: 120.0 cm/s (10.97 Hz)
Rate ratio high/low speed: 119.32x


In [35]:
fig, ax = plt.subplots(figsize=(7, 4))

speed_bins = np.linspace(speed_min_val, speed_max_val, 30)
bin_idx = np.digitize(spk_cov_df_speed["speed"], speed_bins) - 1  # changed

obs_speed, obs_rate, obs_ci = [], [], []
for b in range(len(speed_bins) - 1):
    sel = bin_idx == b
    if sel.sum() > 50:
        counts = spk_cov_df_speed.loc[sel, "spike_count"].values  # changed
        obs_speed.append(speed_bins[b:b+2].mean())
        obs_rate.append(counts.mean() / BIN_SIZE)
        obs_ci.append(1.96 * stats.sem(counts) / BIN_SIZE)

obs_speed, obs_rate, obs_ci = map(np.array, [obs_speed, obs_rate, obs_ci])
ax.errorbar(obs_speed, obs_rate, yerr=obs_ci,
            fmt="o", ms=4, color="grey", ecolor="lightgrey",
            elinewidth=1.5, capsize=3, label="observed ± 95% CI", zorder=3)

speed_range = np.linspace(speed_min_val, speed_max_val, 300)
speed_range_scaled = (speed_range - speed_min_val) / (speed_max_val - speed_min_val)
pred_rate = results_speed_spline.predict(pd.DataFrame({"speed_scaled": speed_range_scaled})) / BIN_SIZE

ax.plot(speed_range, pred_rate, color="steelblue", lw=2, label=f"spline fit (df={bs_df})")
ax.set_xlabel("speed (cm/s)")
ax.set_ylabel("firing rate (Hz)")
ax.set_title(f"unit {unit_idx} — speed tuning")
ax.legend()
sns.despine()
plt.tight_layout()

#### Fit on all units

In [ ]:
# speed_scaled already in cov_df_common (added in common mask cell)
# speed_spline_model_all = fit_glm_all_units(f"spike_count ~ bs(speed_scaled, df={bs_df})",
#                                             cov_df_common, spike_counts_common, unit_ids)

In [36]:
# speed_spline_model_all["model"] = "speed spline"
# speed_spline_model_all.to_csv(f"{base_dir}/analysis/speed_spline_model_all.csv")
speed_spline_model_all = pd.read_csv(f"{base_dir}/analysis/speed_spline_model_all.csv", index_col=0)
speed_spline_model_all.head(1)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,109341.498515416,-54665.749257708,87907.224402639,5.000000000,1132036.000000000,True,"{'Intercept': -2.7247468518001967, 'bs(speed_s...","{'Intercept': 0.07755516483500097, 'bs(speed_s...",100002.515597235,4.000000000,NaN,speed_spline


### Linear position (spline)

#### Fit on one unit

In [46]:
# pos_min_val, pos_max_val, pos_scaled defined in common mask cell
model_pos_spline = smf.glm("spike_count ~ bs(pos_scaled, df=8)",
                           data=spk_cov_df_common,
                           family=sm.families.Poisson())
results_pos_spline = model_pos_spline.fit()
print(results_pos_spline.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1132036
Model:                            GLM   Df Residuals:                  1132027
Model Family:                 Poisson   Df Model:                            8
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -12070.
Date:                Thu, 12 Mar 2026   Deviance:                       20882.
Time:                        12:01:26   Pearson chi2:                 1.13e+06
No. Iterations:                     9   Pseudo R-squ. (CS):          0.0003864
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

In [47]:
pos_range = np.linspace(pos_min_val, pos_max_val, 500)
pos_range_scaled = (pos_range - pos_min_val) / (pos_max_val - pos_min_val)  # changed
pred_rate = results_pos_spline.predict(pd.DataFrame({"pos_scaled": pos_range_scaled})) / BIN_SIZE  # changed

peak_pos  = pos_range[np.argmax(pred_rate)]
peak_rate = pred_rate.max()
print(f"Peak firing position: {peak_pos:.1f} cm, rate: {peak_rate:.2f} Hz")

Peak firing position: 433.2 cm, rate: 1.83 Hz


In [48]:
fig, ax = plt.subplots()

pos_bins = np.linspace(pos_min_val, pos_max_val, 40)
bin_idx  = np.digitize(spk_cov_df["linear_position"].dropna(), pos_bins) - 1
pos_df   = spk_cov_df[spk_cov_df["linear_position"].notna()].copy()

obs_pos, obs_rate, obs_ci = [], [], []
for b in range(len(pos_bins) - 1):
    sel = bin_idx == b
    if sel.sum() > 50:
        counts = pos_df.iloc[np.where(sel)[0]]["spike_count"].values
        obs_pos.append(pos_bins[b:b+2].mean())
        obs_rate.append(counts.mean() / BIN_SIZE)
        obs_ci.append(1.96 * stats.sem(counts) / BIN_SIZE)

ax.errorbar(obs_pos, obs_rate, yerr=obs_ci,
            fmt="o", ms=4, color="grey", ecolor="lightgrey",
            elinewidth=1.5, capsize=3, label="observed ± 95% CI", zorder=3)

ax.plot(pos_range, pred_rate, color="steelblue", lw=2, label="spline fit (df=8)")
ax.axvline(peak_pos, color="red", lw=1, linestyle="--", label=f"peak @ {peak_pos:.0f} cm")
ax.set_xlabel("linear position (cm)")
ax.set_ylabel("firing rate (Hz)")
ax.set_title(f"unit {unit_idx} — position tuning")
ax.legend()
sns.despine()
plt.tight_layout()

In [49]:
graph = sgpl.TrackGraph & {"track_graph_name": "Wtrack_wilbur20210512"}

# Use original camera frames (not 2ms-interpolated bins) — avoids diagonal artifacts
# at trial transitions where np.interp crosses track space (changed)
pos_run = trialized_position[trialized_position["zone"] == "run"][
    ["linear_position", "projected_x_position", "projected_y_position"]
].dropna().iloc[::5]

pos_run = pos_run.copy()
pos_run["pos_scaled"] = (pos_run["linear_position"] - pos_min_val) / (pos_max_val - pos_min_val)
pos_run = pos_run[(pos_run["pos_scaled"] >= 0) & (pos_run["pos_scaled"] <= 1)]

track_rate_hz = results_pos_spline.predict(pos_run[["pos_scaled"]]) / BIN_SIZE

fig, (ax_curve, ax_track) = plt.subplots(1, 2, figsize=(14, 5))

ax_curve.plot(pos_range, pred_rate, color="steelblue", lw=2)
ax_curve.errorbar(obs_pos, obs_rate, yerr=obs_ci,
            fmt="o", ms=4, color="grey", ecolor="lightgrey",
            elinewidth=1.5, capsize=3, label="observed ± 95% CI", zorder=3)
ax_curve.set_xlabel("linear position (cm)")
ax_curve.set_ylabel("firing rate (Hz)")
ax_curve.set_title(f"unit {unit_idx} — position tuning")

graph.plot_track_graph(ax=ax_track, draw_edge_labels=False)
for ln in ax_track.lines:
    ln.set_color("lightgrey")

sc = ax_track.scatter(pos_run["projected_x_position"], pos_run["projected_y_position"],
                      c=track_rate_hz, cmap="hot_r", s=4, zorder=3,
                      vmin=track_rate_hz.min(), vmax=track_rate_hz.max())
plt.colorbar(sc, ax=ax_track, label="firing rate (Hz)")
ax_track.set_xlabel("x position (cm)")
ax_track.set_ylabel("y position (cm)")
ax_track.set_title(f"unit {unit_idx} — rate on track")

sns.despine()
plt.tight_layout()

#### Fit on all units

In [ ]:
# pos_scaled already in cov_df_common (added in common mask cell)
# pos_spline_model_all = fit_glm_all_units("spike_count ~ bs(pos_scaled, df=8)",
#                                           cov_df_common, spike_counts_common, unit_ids)

In [50]:
# pos_spline_model_all["model"] = "pos_spline"
# pos_spline_model_all.to_csv(f"{base_dir}/analysis/pos_spline_model_all.csv")
pos_spline_model_all = pd.read_csv(f"{base_dir}/analysis/pos_spline_model_all.csv", index_col = 0)
pos_spline_model_all.head(2)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,111963.961503106,-55972.980751553,90521.687390328,9.000000000,1132036.000000000,True,"{'Intercept': -3.435351682548603, 'bs(pos_scal...","{'Intercept': 0.06251283062164546, 'bs(pos_sca...",100002.515597235,8.000000000,NaN,pos_spline
1,1,49076.701050337,-24529.350525169,41503.473639060,9.000000000,1132036.000000000,True,"{'Intercept': -4.908393140928331, 'bs(pos_scal...","{'Intercept': 0.12527615997502198, 'bs(pos_sca...",43103.631638410,8.000000000,NaN,pos_spline


## Model comparison

In [75]:
from scipy.stats import chi2

# Load all model CSVs
model_files = {
    "null":         (f"{base_dir}/analysis/null_model_all.csv",        dict(keep_default_na=False, na_values=[""])),
    "null_out":     (f"{base_dir}/analysis/null_model_out_all.csv",    dict(keep_default_na=False, na_values=[""])),
    "trial_type":   (f"{base_dir}/analysis/trial_type_model_all.csv",  {}),
    "choice":       (f"{base_dir}/analysis/choice_model_all.csv",      {}),
    "speed":        (f"{base_dir}/analysis/speed_model_all.csv",       {}),
    "speed_spline": (f"{base_dir}/analysis/speed_spline_model_all.csv",{}),
    "pos_spline":   (f"{base_dir}/analysis/pos_spline_model_all.csv",  {}),
}

models = {}
for name, (path, kwargs) in model_files.items():
    df = pd.read_csv(path, index_col=0, **kwargs)
    df["model"] = name
    models[name] = df.set_index("unit")

# Null baseline per model — must match the dataset the model was fit on
null_for = {
    "trial_type":   "null",
    "speed":        "null",
    "speed_spline": "null",
    "pos_spline":   "null",
    "choice":       "null_out",
}

rows = []
for model_name, null_name in null_for.items():
    print(model_name)
    m   = models[model_name]
    nul = models[null_name]

    for uid in m.index:
        row  = m.loc[uid]
        null_row = nul.loc[uid]

        lrt_stat = 2 * (row["llf"] - null_row["llf"])
        lrt_df   = int(row["df_model"])
        lrt_pval = (1 - chi2.cdf(lrt_stat, lrt_df)) if lrt_df > 0 else np.nan

        rows.append(dict(
            unit        = uid,
            model       = model_name,
            aic         = row["aic"],
            llf         = row["llf"],
            n_params    = row["n_params"],
            n_obs       = row["n_obs"],
            converged   = row["converged"],
            delta_aic   = row["aic"] - null_row["aic"],
            lrt_stat    = lrt_stat,
            lrt_df      = lrt_df,
            lrt_pval    = lrt_pval,
            tuned       = bool(lrt_pval < 0.05) if not np.isnan(lrt_pval) else False,
        ))

comparison = pd.DataFrame(rows)

summary = comparison.groupby("model").agg(
    mean_dAIC  = ("delta_aic", "mean"),
    median_dAIC= ("delta_aic", "median"),
    n_tuned    = ("tuned",     "sum"),
    frac_tuned = ("tuned",     "mean"),
    n_converged= ("converged", "sum"),
).round(3)
print(summary)

trial_type


ValueError: cannot convert float NaN to integer

In [ ]:
# Pivot to wide AIC table — enables direct pairwise model comparison
aic_wide = comparison.pivot(index="unit", columns="model", values="aic")

# Add null baselines (same common dataset, so directly comparable)
aic_wide["null"]     = models["null"]["aic"]
aic_wide["null_out"] = models["null_out"]["aic"]

# Best single-variable model per unit (all-trials models only)
all_trials_models = ["trial_type", "speed", "speed_spline", "pos_spline"]
aic_wide["best_model"] = aic_wide[all_trials_models].idxmin(axis=1)
print("Best model counts (all units):")
print(aic_wide["best_model"].value_counts())

# LRT: speed_linear vs speed_spline (nested — spline adds 3 df)
llf_wide = comparison.pivot(index="unit", columns="model", values="llf")
lrt_speed = 2 * (llf_wide["speed_spline"] - llf_wide["speed"])
lrt_speed_pval = lrt_speed.apply(lambda x: 1 - chi2.cdf(x, df=3))  # Δdf = 5-2 = 3
n_nonlinear = (lrt_speed_pval < 0.05).sum()
print(f"\nSpeed: nonlinear > linear in {n_nonlinear}/{len(lrt_speed_pval)} units (LRT p<0.05)")

In [53]:
out = cov_df["trial_type"] == "outbound"                                                                
print(f"outbound bins total:          {out.sum():,}")
print(f"  + speed.notna():            {(out & cov_df['speed'].notna()).sum():,}")                       
print(f"  + speed > 5:                {(out & cov_df['speed'].notna() & (cov_df['speed'] > 5)).sum():,}")
print(f"  + speed < 120:              {(out & cov_df['speed'].notna() & (cov_df['speed'] > 5) &(cov_df['speed'] < 120)).sum():,}")
print(f"  + pos.notna():              {(out & common_mask).sum():,}")



outbound bins total:          867,583
  + speed.notna():            203,076
  + speed > 5:                194,886
  + speed < 120:              194,672
  + pos.notna():              194,672


In [59]:
print(((cov_df["zone"]=="run")).sum())
print(((cov_df["zone"]=="run") & (cov_df["trial_type"]=="outbound")).sum())

1835243
867583


In [66]:
cov_df[((cov_df["zone"]=="run") & (cov_df["trial_type"]=="outbound"))]["speed"].notna().sum()

203076

In [72]:
(trialized_position[trialized_position["trial_type"]=="outbound"])["speed"]

time
1620843619.282530785    3.696854205
1620843619.297107697    3.745092960
1620843619.311684608    3.820923852
1620843619.326261520    3.907659278
1620843619.340838432    3.990089036
                           ...     
1620851244.096884489   22.450380894
1620851244.111461401   21.495332557
1620851244.126038551   20.758213655
1620851244.140615702   20.203871475
1620851244.155192375   19.790271726
Name: speed, Length: 81651, dtype: float64

In [80]:
(trial_type_model_all["df_model"].notna()).sum()

262